# Семинар 12. HTTP-клиенты и управляемая конкурентность

Это справочное занятие для проектов. Сначала разберём один настоящий ответ GitHub API, затем выполним несколько запросов конкурентно и отдельно воспроизведём тайм-аут, отмену и ограничение параллелизма без зависимости от сети.

## Цели

После семинара вы сможете:

- диагностировать HTTP-запрос по URL, статусу, заголовкам и телу;
- разделять транспорт, декодирование JSON и проверку предметной схемы;
- пользоваться `requests` с тайм-аутом;
- организовывать конкурентные операции через `TaskGroup`;
- выполнять HTTP-запросы через общую `aiohttp.ClientSession`;
- воспроизводить тайм-аут и отмену локально;
- проверять фактический предел semaphore;
- находить блокирующий вызов в async-коде;
- переносить примеры в проект без привязки предметной логики к сети.

## Перед началом

Нужен Python 3.14 и зависимости из `requirements.txt`. Сетевые ячейки требуют интернета и помечены `network`; при ответе `403` или `429` остановите повторные запросы и продолжайте с локальными упражнениями. Семинар рассчитан примерно на 90 минут обсуждения и воспроизведения, но обязательной сдаваемой работы нет.

В Jupyter уже запущен event loop, поэтому примеры используют top-level `await`. В обычном `.py`-скрипте вызывайте одну верхнеуровневую coroutine через `asyncio.run(main())`.

## Минимальный чек-лист HTTP-клиента

Перед разбором тела ответьте:

1. Какой метод и URL отправлены?
2. Какой timeout задан?
3. Получен ли ответ и каков status?
4. Какой `Content-Type` объявлен?
5. Нужно ли вызвать `raise_for_status()`?
6. Соответствует ли разобранный JSON ожидаемой схеме?
7. Можно ли безопасно повторить операцию?

Логировать токен `Authorization`, cookie и персональные данные нельзя. Для диагностики обычно достаточно метода, host/path, статуса, длительности и идентификатора запроса.

## Один настоящий запрос

Получим сведения о CPython. `User-Agent` явно называет учебный клиент, timeout не позволяет ждать бесконечно, а `raise_for_status()` отделяет успешный статус от `4xx/5xx`. Количество звёзд меняется со временем, поэтому проверяем форму, а не конкретное число.

In [ ]:
import requests

response = requests.get(
    "https://api.github.com/repos/python/cpython",
    headers={"Accept": "application/vnd.github+json", "User-Agent": "hse-python-course"},
    timeout=10,
)
print(response.status_code, response.headers.get("content-type"))
response.raise_for_status()
payload = response.json()
print(payload["full_name"], payload["stargazers_count"], payload["language"])

Ошибки принадлежат разным слоям:

- `requests.Timeout` — превышено время ожидания;
- `requests.ConnectionError` — не удалось установить или сохранить соединение;
- `requests.HTTPError` — `raise_for_status()` увидел `4xx/5xx`;
- `requests.JSONDecodeError` — тело не оказалось корректным JSON;
- `ValueError` нашего валидатора — JSON корректен, но предметная схема неверна.

Не обязательно ловить их все в каждой функции. Важно не стирать различие пустым `except Exception: return None`.

## Упражнение 1. Валидация ответа без сети

Напишите `repo_summary(payload)`. Корень должен быть словарём, `full_name` — строкой, `stargazers_count` — целым неотрицательным числом, но не `bool`, `language` — строкой или `None`. Верните новый словарь только с этими полями. На первой проблеме поднимите `ValueError` с именем поля.

In [ ]:
def repo_summary(payload: object) -> dict:
    ...

assert repo_summary({
    "full_name": "python/cpython",
    "stargazers_count": 70_000,
    "language": "Python",
}) == {
    "full_name": "python/cpython",
    "stars": 70_000,
    "language": "Python",
}

try:
    repo_summary({"full_name": "x/y", "stargazers_count": True, "language": None})
except ValueError as error:
    assert "stargazers_count" in str(error)
else:
    raise AssertionError("bool must not be accepted as stars")

## Управляемая модель ожидания

Прежде чем добавлять сеть, сравним два расписания на `asyncio.sleep`. Это не макет HTTP-данных, а управляемый источник задержки: он позволяет увидеть время без DNS, лимитов и случайных ответов. Та же структура сохранится при замене `sleep` на `session.get`.

In [ ]:
import asyncio
from time import perf_counter

async def operation(name: str, delay: float) -> str:
    await asyncio.sleep(delay)
    return name

async def sequential() -> list[str]:
    return [await operation("slow", 0.06), await operation("fast", 0.02)]

async def concurrent() -> list[str]:
    async with asyncio.TaskGroup() as group:
        slow = group.create_task(operation("slow", 0.06))
        fast = group.create_task(operation("fast", 0.02))
    return [slow.result(), fast.result()]

start = perf_counter()
await sequential()
sequential_time = perf_counter() - start
start = perf_counter()
await concurrent()
concurrent_time = perf_counter() - start
print(round(sequential_time, 3), round(concurrent_time, 3))

> **Появилось в Python 3.11.** `TaskGroup` ждёт все дочерние задачи при выходе и отменяет соседей при ошибке. В Python 3.14 его `create_task()` дополнительно передаёт все keyword-аргументы нижележащему `loop.create_task()`. Для упражнения важна базовая гарантия времени жизни, а не новая низкоуровневая настройка.

## Упражнение 2. Конкурентный запуск зависимости

Реализуйте `run_all(keys, worker)`: создайте все задачи в одном `TaskGroup`, верните результаты в порядке `keys`. `worker` — переданная async-функция. Затем проверьте функцию на локальной зависимости с разными задержками. Не используйте `gather`.

In [ ]:
from collections.abc import Awaitable, Callable
from typing import TypeVar

T = TypeVar("T")
R = TypeVar("R")

async def run_all(keys: list[T], worker: Callable[[T], Awaitable[R]]) -> list[R]:
    ...

async def fake_worker(key: str) -> str:
    delays = {"a": 0.03, "b": 0.01, "c": 0.02}
    await asyncio.sleep(delays[key])
    return key.upper()

assert await run_all(["a", "b", "c"], fake_worker) == ["A", "B", "C"]

## Несколько репозиториев через `aiohttp`

Одна session обслуживает всю серию. Функция одного запроса получает session снаружи: она не владеет пулом соединений и не закрывает его. После выхода из внутреннего `async with` тело прочитано, а соединение возвращено session. После выхода из внешнего блока закрывается сама session.

In [ ]:
import aiohttp

async def fetch_repo(session: aiohttp.ClientSession, name: str) -> dict:
    async with session.get(f"https://api.github.com/repos/{name}") as response:
        response.raise_for_status()
        return repo_summary(await response.json())

async def fetch_repos(names: list[str]) -> list[dict]:
    timeout = aiohttp.ClientTimeout(total=10)
    headers = {"Accept": "application/vnd.github+json", "User-Agent": "hse-python-course"}
    async with aiohttp.ClientSession(timeout=timeout, headers=headers) as session:
        async with asyncio.TaskGroup() as group:
            tasks = [group.create_task(fetch_repo(session, name)) for name in names]
    return [task.result() for task in tasks]

repositories = await fetch_repos(["python/cpython", "psf/requests", "aio-libs/aiohttp"])
for repository in repositories:
    print(repository)

## Общий тайм-аут пакета

`aiohttp.ClientTimeout` ограничивает поведение клиента. Иногда предметное правило звучит шире: «получить весь отчёт не дольше двух секунд». Тогда весь `TaskGroup` помещают внутрь `asyncio.timeout(2)`.

> **Появилось в Python 3.11.** `asyncio.timeout()` — асинхронный контекстный менеджер. После истечения срока он отменяет текущую задачу внутри блока и снаружи поднимает встроенный `TimeoutError`.

In [ ]:
async def bounded_operation() -> str:
    try:
        async with asyncio.timeout(0.02):
            return await operation("too slow", 0.2)
    except TimeoutError:
        return "timeout"

assert await bounded_operation() == "timeout"

## Отмена и очистка

Отмена приходит в ожидающую coroutine как `CancelledError`. Ресурс освобождаем в `finally`; саму отмену обычно не ловим либо после дополнительной работы поднимаем снова. Следующий пример записывает наблюдаемый порядок событий.

In [ ]:
events = []

async def long_operation() -> None:
    events.append("started")
    try:
        await asyncio.sleep(10)
    finally:
        events.append("cleaned")

task = asyncio.create_task(long_operation())
await asyncio.sleep(0)
task.cancel()
try:
    await task
except asyncio.CancelledError:
    events.append("cancelled outside")

assert events == ["started", "cleaned", "cancelled outside"]

Если задача внутри `TaskGroup` падает, группа отменяет незавершённых соседей. Исключения после очистки объединяются в `ExceptionGroup`. Не превращайте каждую ошибку в `None`, если пакет должен быть атомарным: частичный безымянный результат сложнее корректно использовать, чем явная ошибка.

In [ ]:
async def fail() -> None:
    await asyncio.sleep(0.01)
    raise ValueError("bad response schema")

try:
    async with asyncio.TaskGroup() as group:
        slow_task = group.create_task(operation("slow", 1))
        failed_task = group.create_task(fail())
except* ValueError as error_group:
    assert len(error_group.exceptions) == 1

assert slow_task.cancelled()

## Проверяем semaphore измерением

Фраза «у нас limit=3» должна подтверждаться тестом. Под защитой одного lock будем считать число задач внутри worker и запоминать максимум. Сам worker входит в semaphore через `async with`, поэтому разрешение возвращается и при отмене.

In [ ]:
active = 0
max_active = 0
counter_lock = asyncio.Lock()

async def measured_worker(item: int, semaphore: asyncio.Semaphore) -> int:
    global active, max_active
    async with semaphore:
        async with counter_lock:
            active += 1
            max_active = max(max_active, active)
        try:
            await asyncio.sleep(0.01)
            return item * 2
        finally:
            async with counter_lock:
                active -= 1

semaphore = asyncio.Semaphore(3)
async with asyncio.TaskGroup() as group:
    tasks = [group.create_task(measured_worker(item, semaphore)) for item in range(10)]
assert [task.result() for task in tasks] == [item * 2 for item in range(10)]
assert max_active == 3 and active == 0

## Упражнение 3. `map_limited`

Обобщите предыдущий пример. Функция получает `items`, async-функцию `worker` и положительный `limit`. Один semaphore ограничивает вход в worker, один `TaskGroup` отвечает за время жизни задач, а результат сохраняет порядок входа. Для `limit <= 0` поднимите `ValueError` до создания задач.

In [ ]:
async def map_limited(
    items: list[T],
    worker: Callable[[T], Awaitable[R]],
    limit: int,
) -> list[R]:
    ...

async def double(item: int) -> int:
    await asyncio.sleep(0.01)
    return item * 2

assert await map_limited([1, 2, 3, 4], double, 2) == [2, 4, 6, 8]
try:
    await map_limited([1], double, 0)
except ValueError:
    pass
else:
    raise AssertionError("limit=0 must be rejected")

## Найдите блокировку event loop

В async-функции ниже `time.sleep` блокирует весь поток, поэтому heartbeat перестанет печататься. Замена на `await asyncio.sleep` исправляет только искусственную задержку. Для настоящего синхронного I/O выбирают async-библиотеку или, если её нет, изолируют короткий вызов через `asyncio.to_thread`. CPU-bound функцию следует профилировать и обычно переносить в процессы.

In [ ]:
import time

async def bad_async_function() -> None:
    time.sleep(1)  # блокирует event loop; не запускайте рядом с важными задачами

# Корректная модель неблокирующего ожидания:
async def good_async_function() -> None:
    await asyncio.sleep(1)

## Чек-лист для группового проекта

- Одна ли долгоживущая session используется на серию запросов?
- Есть ли timeout у каждого внешнего ожидания или у всей операции?
- Проверяется ли HTTP-статус до разбора результата?
- Валидируется ли предметная схема отдельно от JSON?
- Ограничено ли число конкурентных запросов?
- Освобождаются ли ресурсы в `async with` или `finally`?
- Не находится ли внутри async-кода `requests`, `time.sleep` или тяжёлый цикл?
- Сохраняет ли функция фоновые задачи без владельца?
- Можно ли протестировать оркестрацию с fake worker без интернета?
- Не попадают ли секреты в лог и traceback?

Если проект делает только один запрос по нажатию кнопки, синхронный клиент может быть проще и достаточнее. Async нужен из поведения нагрузки, а не из желания добавить технологию.

## Самопроверка

1. Какие пять частей HTTP-обмена надо проверить?
2. Чем `HTTPError` отличается от `ConnectionError`?
3. Почему разбор JSON не заменяет валидацию?
4. Почему тест не должен зависеть от GitHub API?
5. Что ограничивает время жизни задач в `TaskGroup`?
6. Почему результаты можно вернуть во входном порядке, хотя завершались они иначе?
7. Зачем session передаётся в функцию одного запроса?
8. Как общий timeout связан с отменой?
9. Где должна находиться очистка coroutine?
10. Как проверить предел semaphore?
11. Почему `time.sleep` блокирует соседние coroutine?
12. Когда синхронный HTTP-клиент лучше асинхронного?

## Итоги

- HTTP-клиент отдельно отвечает за доставку, статус и декодирование, предметный код — за схему данных.
- Внешние запросы всегда имеют границу времени.
- `TaskGroup` структурирует конкурентную работу и отменяет соседей при ошибке.
- Одна `aiohttp.ClientSession` обслуживает серию запросов и повторно использует соединения.
- `asyncio.timeout()` появился в Python 3.11 и использует отмену внутри блока.
- `CancelledError` не скрывают, а ресурсы освобождают в `finally`.
- Semaphore ограничивает фактическое число одновременно выполняющихся операций.
- Fake worker делает async-логику воспроизводимой без сети.

Домашняя работа закрепляет URL, проверку JSON, `TaskGroup`, общий timeout, semaphore и корректную очистку задач.